# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os
import json
import time
import chromadb
from typing import Any, Dict, List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb.utils import embedding_functions
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import AIMessage
from lib.tooling import tool


In [3]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

load_dotenv(".env")
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY") or OPENAI_API_KEY
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")
MAX_RETRIES = 10

os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
assert OPENAI_API_KEY, "OPENAI_API_KEY is required"
assert CHROMA_OPENAI_API_KEY, "CHROMA_OPENAI_API_KEY or OPENAI_API_KEY is required"


In [4]:
CHROMA_PATH = "chromadb"
COLLECTION_NAME = "udaplay"
MODEL_NAME = "gpt-4o-mini"

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
    model_name="text-embedding-3-small",
)
embedding_fn.client = embedding_fn.client.with_options(max_retries=MAX_RETRIES)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY) if TAVILY_API_KEY else None
long_term_memory = []

def retry_call(fn, max_retries=MAX_RETRIES, delay=1):
    for attempt in range(1, max_retries + 1):
        try:
            return fn()
        except Exception:
            if attempt == max_retries:
                raise
            time.sleep(delay)
            delay = min(delay * 2, 20)


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query: str) -> List[Dict[str, Any]]:
    """Semantic search: Finds most results in the vector DB"""
    results = retry_call(lambda: collection.query(
        query_texts=[query],
        n_results=5,
        include=["documents", "metadatas", "distances"],
    ))
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    return [
        {
            "Name": meta.get("Name"),
            "Platform": meta.get("Platform"),
            "Genre": meta.get("Genre"),
            "Publisher": meta.get("Publisher"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
            "distance": distance,
            "source": "local_vector_db",
            "document": doc,
        }
        for doc, meta, distance in zip(docs, metas, distances)
    ]


#### Evaluate Retrieval Tool

In [6]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether retrieved documents answer the question")
    confidence: float = Field(description="Confidence from 0 to 1", ge=0, le=1)
    description: str = Field(description="Short reason")

@tool
def evaluate_retrieval(question: str, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Evaluate retrieved documents for a question"""
    if not retrieved_docs:
        return EvaluationReport(
            useful=False,
            confidence=0.0,
            description="No local documents were retrieved."
        ).model_dump()

    question_terms = {term.lower().strip("?.,!") for term in question.split() if len(term) > 2}
    text = json.dumps(retrieved_docs, ensure_ascii=False).lower()
    matches = sum(1 for term in question_terms if term in text)
    confidence = min(1.0, matches / max(1, min(len(question_terms), 6)))
    useful = confidence >= 0.35 or any(doc.get("distance", 1.0) < 0.45 for doc in retrieved_docs)

    report = EvaluationReport(
        useful=useful,
        confidence=round(confidence, 2),
        description=(
            "Local results appear sufficient." if useful
            else "Local results are weak; use web search."
        ),
    )
    return report.model_dump()


#### Game Web Search Tool

In [7]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str) -> List[Dict[str, Any]]:
    """Search the web for game industry information"""
    if tavily_client is None:
        return [{"error": "TAVILY_API_KEY is required for web search."}]

    response = retry_call(lambda: tavily_client.search(
        query=question,
        search_depth="basic",
        max_results=5,
        include_answer=True,
    ))
    results = [
        {
            "title": item.get("title"),
            "url": item.get("url"),
            "content": item.get("content"),
            "source": "tavily_web_search",
        }
        for item in response.get("results", [])
    ]
    long_term_memory.append({"question": question, "results": results})
    return results


### Agent

In [8]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

instructions = """
You are UdaPlay, an AI research agent for the video game industry.
Always start with retrieve_game for internal knowledge.
Then call evaluate_retrieval on the retrieved results.
If local results are weak, call game_web_search.
Give a concise final answer with sources.
Mention whether evidence came from local_vector_db or web search.
"""

udaplay_agent = Agent(
    model_name=MODEL_NAME,
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)


In [9]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

example_queries = [
    "When were Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for i, query in enumerate(example_queries, start=1):
    print("\nQUERY:", query)
    run = udaplay_agent.invoke(query, session_id=f"demo-{i}")
    final_state = run.get_final_state()
    final_answer = None
    used_tools = []
    for message in final_state["messages"]:
        if isinstance(message, AIMessage):
            if message.tool_calls:
                used_tools.extend(call.function.name for call in message.tool_calls)
            elif message.content:
                final_answer = message.content
    print("TOOLS:", used_tools)
    print("ANSWER:", final_answer)



QUERY: When were Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
TOOLS: ['retrieve_game', 'evaluate_retrieval']
ANSWER: Pokémon Gold and Silver were released in 1999 for the Game Boy Color. This information is sourced from the local vector database.

QUERY: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


...

[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
TOOLS: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
ANSWER: Mortal Kombat X was not originally released for PlayStation 5, but it is compatible with the console. The game was initially launched for PlayStation 4, Xbox One, and PC in 2015. However, it can be played on PlayStation 5 due to backward compatibility. Additionally, an upgraded version called Mortal Kombat XL, which includes all downloadable content, is also available for PlayStation 5.

Sources:
- [Mortal Kombat X - Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)
- [Mortal Kombat X | GameStop](https://www.gamestop.com/video-games/products/mortal-kombat-x/11003580.html)
- [Mortal Kombat XL - PlayStation Store](https://store.playstation.com/en-us/product/UP1018-CUSA03589_00-MORTALKOMBATXL00)

This information was obtained from a web search.


### (Optional) Advanced

In [10]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

print(f"Long-term memory entries: {len(long_term_memory)}")
for session_id in ["demo-1", "demo-2", "demo-3"]:
    print(f"{session_id} runs: {len(udaplay_agent.get_session_runs(session_id))}")


Long-term memory entries: 1
demo-1 runs: 1
demo-2 runs: 1
demo-3 runs: 1
